# Daily Challenge: Building Trustworthy Insights with BERT

**Goal:** Fine-tune DistilBERT on tweet sentiment, inspect its attention maps, evaluate rigorously, and build an explainable inference helper.

**Dataset:** `tweet_eval` (sentiment) — 3 classes: Negative (0), Neutral (1), Positive (2)

---

## Learning Objectives
| # | Skill | What You Will Build |
|---|-------|---------------------|
| 1 | Data Loading | Class distributions, sample tweets |
| 2 | Tokenization | Batched preprocessing with AutoTokenizer |
| 3 | Fine-Tuning | DistilBERT via Hugging Face Trainer API |
| 4 | Evaluation | Accuracy, F1, confidence histogram |
| 5 | Attention Inspection | CLS heatmap and token attribution |
| 6 | Explainable Inference | `analyze_tweet()` returning label and key tokens |

---
## Environment Setup

Install the required packages. Run this cell once per environment.

In [ ]:
!pip install -q datasets transformers[torch] evaluate accelerate seaborn scikit-learn

In [ ]:
import os
import warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

import torch
import torch.nn.functional as F

from datasets import load_dataset
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModel,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {device.upper()}")

---
## Task 1 — Data Loading and Inspection

We use `tweet_eval` because it covers real Twitter language and comes pre-split with clean labels.

Check that classes are roughly balanced. A large imbalance would require weighted loss or oversampling.

In [ ]:
dataset     = load_dataset("tweet_eval", "sentiment")
label_names = ["Negative", "Neutral", "Positive"]
num_labels  = len(label_names)

print(dataset)

In [ ]:
bar_colors = ["#E74C3C", "#95A5A6", "#2ECC71"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Class Distribution — tweet_eval (sentiment)", fontsize=14)

for ax, split_name in zip(axes, ["train", "validation", "test"]):
    split  = dataset[split_name]
    counts = Counter(split["label"])
    values = [counts[i] for i in range(3)]
    bars   = ax.bar(label_names, values, color=bar_colors, edgecolor="white")
    ax.set_title(f"{split_name.capitalize()} (n={len(split):,})")
    ax.set_ylabel("Count")
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
                f"{val:,}", ha="center", va="bottom", fontsize=10)
    ax.set_ylim(0, max(values) * 1.2)

plt.tight_layout()
plt.show()
print("Neutral tweets are the most frequent class — mild class imbalance.")
print("We will use macro-F1 alongside accuracy to surface this.")

In [ ]:
# Collect two example tweets per label for later attention visualisation
sample_tweets = {i: [] for i in range(num_labels)}

for row in dataset["train"]:
    lid = row["label"]
    if len(sample_tweets[lid]) < 2:
        sample_tweets[lid].append(row["text"])
    if all(len(v) == 2 for v in sample_tweets.values()):
        break

for label_id, tweets in sample_tweets.items():
    print(f"\n{label_names[label_id].upper()} (label={label_id})")
    for i, tweet in enumerate(tweets, 1):
        print(f"  [{i}] {tweet[:120]}")

---
## Task 2 — Tokenization Pipeline

We use DistilBERT because it is 40% smaller and 60% faster than BERT while keeping ~97% of its performance.

We cap at 128 tokens because almost all tweets fit within that limit.

In [ ]:
model_name = "distilbert-base-uncased"
max_len    = 128

tokenizer = AutoTokenizer.from_pretrained(model_name)
print(f"Tokenizer:  {tokenizer.__class__.__name__}")
print(f"Vocab size: {tokenizer.vocab_size:,}")
print(f"Max length: {max_len} tokens")

In [ ]:
def tokenize_batch(batch):
    encoded = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=max_len,
    )
    encoded["labels"] = batch["label"]
    return encoded


tokenized = dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text", "label"],
)

tokenized = tokenized.shuffle(seed=42)
tokenized.set_format("torch")

print(tokenized)
print("Sample keys:", list(tokenized["train"][0].keys()))
print("input_ids shape:", tokenized["train"][0]["input_ids"].shape)

In [ ]:
sample     = tokenized["train"][7]
all_tokens = tokenizer.convert_ids_to_tokens(sample["input_ids"].tolist())

real_tokens = [
    tok
    for tok, mask in zip(all_tokens, sample["attention_mask"].tolist())
    if mask == 1
]

print(f"Label: {label_names[sample['labels'].item()]}")
print(f"Real tokens ({len(real_tokens)}):", real_tokens)

---
## Task 3 — Fine-Tuning with the Trainer API

Key hyperparameters:
| Setting | Value | Reason |
|---------|-------|--------|
| `learning_rate` | 5e-5 | Standard range for BERT fine-tuning |
| `batch_size` | 32 | Fits a 16 GB GPU |
| `num_epochs` | 3 | Enough for convergence on ~45k examples |
| `weight_decay` | 0.01 | Prevents overfitting on the classifier head |
| `load_best_model_at_end` | True | Saves the best validation checkpoint |

In [ ]:
acc_metric = evaluate.load("accuracy")
f1_metric  = evaluate.load("f1")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = acc_metric.compute(predictions=preds, references=labels)
    f1  = f1_metric.compute(predictions=preds, references=labels, average="macro")
    return {"accuracy": acc["accuracy"], "f1_macro": f1["f1"]}


print("compute_metrics ready.")

In [ ]:
output_dir = "./distilbert-tweet-sentiment"

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label={i: name for i, name in enumerate(label_names)},
    label2id={name: i for i, name in enumerate(label_names)},
)

train_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_strategy="steps",
    logging_steps=50,
    report_to="none",
    seed=42,
)

collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")

In [ ]:
# Expected time: ~8 min on a T4 GPU, ~45 min on CPU.
print("Starting fine-tuning...")
train_result = trainer.train()
print("Training complete!")
print(f"Steps : {train_result.global_step}")
print(f"Loss  : {train_result.training_loss:.4f}")

In [ ]:
save_dir = "./distilbert-tweet-sentiment-best"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"Model saved to '{save_dir}'")
print("Files:", os.listdir(save_dir))

In [ ]:
log_history = trainer.state.log_history

train_logs = [l for l in log_history if "loss" in l and "eval_loss" not in l]
eval_logs  = [l for l in log_history if "eval_loss" in l]

train_steps  = [l["step"]           for l in train_logs]
train_losses = [l["loss"]           for l in train_logs]
eval_epochs  = [l["epoch"]          for l in eval_logs]
eval_losses  = [l["eval_loss"]      for l in eval_logs]
eval_acc     = [l["eval_accuracy"]  for l in eval_logs]
eval_f1      = [l["eval_f1_macro"]  for l in eval_logs]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("DistilBERT Fine-Tuning — Learning Curves", fontsize=14)

axes[0].plot(train_steps, train_losses, label="Train loss", color="#3498DB")
axes[0].plot([l["step"] for l in eval_logs], eval_losses,
             "o--", label="Val loss", color="#E74C3C")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss")
axes[0].legend()

axes[1].plot(eval_epochs, eval_acc, "s-", color="#2ECC71", linewidth=2, markersize=8)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Validation Accuracy")
axes[1].set_ylim(0.5, 1.0)

axes[2].plot(eval_epochs, eval_f1, "D-", color="#9B59B6", linewidth=2, markersize=8)
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Macro F1")
axes[2].set_title("Validation Macro F1")
axes[2].set_ylim(0.5, 1.0)

plt.tight_layout()
plt.show()

best_epoch = eval_epochs[int(np.argmax(eval_f1))]
print(f"Best macro F1 ({max(eval_f1):.4f}) at epoch {best_epoch}.")

---
## Task 4 — Evaluation and Calibration

A model can be accurate but poorly calibrated — predicting 95% confidence when it is correct only 70% of the time.
A calibration histogram shows whether the model's confidence scores are trustworthy.

In [ ]:
val_metrics = trainer.evaluate(tokenized["validation"])

print("VALIDATION RESULTS")
print(f"  Accuracy : {val_metrics['eval_accuracy']:.4f}")
print(f"  Macro F1 : {val_metrics['eval_f1_macro']:.4f}")
print(f"  Loss     : {val_metrics['eval_loss']:.4f}")

if val_metrics["eval_accuracy"] >= 0.70:
    print("Accuracy exceeds 70% — reasonable for a 3-class tweet task.")
else:
    print("Accuracy below 70% — consider more epochs or learning rate tuning.")

In [ ]:
pred_output = trainer.predict(tokenized["test"])
test_logits = pred_output.predictions
test_labels = pred_output.label_ids

test_probs = torch.softmax(torch.tensor(test_logits), dim=-1).numpy()
test_preds = np.argmax(test_probs, axis=-1)
confidence = test_probs.max(axis=-1)

test_acc = (test_preds == test_labels).mean()
test_f1  = f1_metric.compute(
    predictions=test_preds, references=test_labels, average="macro"
)["f1"]

print(f"Test Accuracy  : {test_acc:.4f}")
print(f"Test Macro F1  : {test_f1:.4f}")
print(f"Mean confidence: {confidence.mean():.3f}")

In [ ]:
correct_mask   = (test_preds == test_labels)
correct_conf   = confidence[correct_mask]
incorrect_conf = confidence[~correct_mask]

bins = np.arange(0.0, 1.05, 0.1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Confidence Score Distribution (Test Split)", fontsize=14)

axes[0].hist(confidence, bins=bins, color="#3498DB", edgecolor="white")
axes[0].set_title("All Predictions")
axes[0].set_xlabel("Softmax confidence")
axes[0].set_ylabel("Count")
axes[0].axvline(confidence.mean(), color="#E74C3C", linestyle="--",
                label=f"Mean = {confidence.mean():.2f}")
axes[0].legend()

axes[1].hist(correct_conf,   bins=bins, alpha=0.7, label="Correct",   color="#2ECC71", edgecolor="white")
axes[1].hist(incorrect_conf, bins=bins, alpha=0.7, label="Incorrect", color="#E74C3C", edgecolor="white")
axes[1].set_title("Correct vs Incorrect")
axes[1].set_xlabel("Softmax confidence")
axes[1].set_ylabel("Count")
axes[1].legend()

plt.tight_layout()
plt.show()

n_high_conf        = (confidence > 0.9).sum()
n_high_conf_errors = ((confidence > 0.9) & ~correct_mask).sum()
print(f"Predictions with confidence > 0.90 : {n_high_conf}")
print(f"Of those, incorrect                : {n_high_conf_errors} ({100*n_high_conf_errors/max(n_high_conf,1):.1f}%)")

---
## Task 5 — Attention Inspection

We extract last-layer attention weights, average across all heads, and plot how much each token is attended to from the `[CLS]` position — because the classifier reads the `[CLS]` hidden state.

Note: attention weights are not a rigorous explanation of model behaviour, but they give a useful interpretable signal.

In [ ]:
bert_base = AutoModel.from_pretrained(
    save_dir,
    output_attentions=True,
    ignore_mismatched_sizes=True,
)
bert_base.eval()
print("Base model loaded for attention extraction.")


def get_cls_attention(text):
    """
    Returns:
        tokens   : list of WordPiece tokens (including [CLS] and [SEP])
        cls_attn : 1-D array, average attention from [CLS] across all heads
                   in the last transformer layer, normalised to [0, 1]
    """
    inputs = tokenizer(
        text, return_tensors="pt", truncation=True, max_length=max_len
    )
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].tolist())

    with torch.no_grad():
        outputs = bert_base(**inputs)

    last_layer = outputs.attentions[-1]       # (1, heads, seq, seq)
    avg_attn   = last_layer.mean(dim=1)       # (1, seq, seq)
    cls_row    = avg_attn[0, 0, :].numpy()    # attention from [CLS] token
    cls_row    = cls_row / (cls_row.max() + 1e-9)

    return tokens, cls_row


print("get_cls_attention() ready.")

In [ ]:
sentiment_cmaps = {0: "Reds", 1: "Greys", 2: "Greens"}

fig, axes = plt.subplots(3, 1, figsize=(14, 10))
fig.suptitle("CLS Attention Heatmap — last layer, mean across heads", fontsize=14, y=1.01)

top_token_list = []

for ax, (label_id, tweets) in zip(axes, sample_tweets.items()):
    tweet        = tweets[0]
    tokens, attn = get_cls_attention(tweet)

    n      = len([t for t in tokens if t != "[PAD]"])
    tokens = tokens[:n]
    attn   = attn[:n]

    im = ax.imshow(attn[np.newaxis, :], aspect="auto",
                   cmap=sentiment_cmaps[label_id], vmin=0, vmax=1)
    ax.set_xticks(range(n))
    ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=8)
    ax.set_yticks([])
    ax.set_title(f"True label: {label_names[label_id]} — {tweet[:70]}", fontsize=10, loc="left")
    plt.colorbar(im, ax=ax, orientation="vertical", fraction=0.01, pad=0.01)

    content_mask = np.array([t not in ("[CLS]", "[SEP]") for t in tokens])
    ranked       = np.argsort(attn * content_mask)[::-1]
    top_tokens   = [tokens[i] for i in ranked[:3]]
    top_token_list.append((label_names[label_id], top_tokens, tweet[:60]))

plt.tight_layout()
plt.show()

print("\nAttention Insights")
for sentiment, top_toks, snippet in top_token_list:
    print(f"  {sentiment:>8s} | Top-3 tokens: {top_toks}")
    print(f"            | Snippet: {snippet}")

---
## Task 6 — Inference Helper: `analyze_tweet()`

This function returns three things:

| Field | Purpose |
|-------|---------|
| `label` | Route ticket to the correct queue |
| `confidence` | Escalate only when confidence is above a threshold |
| `key_tokens` | Show evidence tokens so agents can trust the decision |

In [ ]:
classifier = AutoModelForSequenceClassification.from_pretrained(save_dir)
classifier.eval()
print(f"Classifier loaded from '{save_dir}'.")


def analyze_tweet(text, top_k=5, attn_threshold=0.3):
    """
    Classify a text and return label, confidence, and key tokens.

    Parameters
    ----------
    text           : input string
    top_k          : number of high-attention tokens to return
    attn_threshold : minimum normalised attention to include a token

    Returns dict with: label, confidence, all_probs, key_tokens, escalate
    """
    inputs = tokenizer(
        text, return_tensors="pt", truncation=True, max_length=max_len
    )
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].tolist())

    with torch.no_grad():
        logits = classifier(**inputs).logits

    probs   = F.softmax(logits, dim=-1)[0]
    pred_id = probs.argmax().item()
    conf    = probs[pred_id].item()
    label   = label_names[pred_id]

    all_probs = {label_names[i]: round(probs[i].item(), 4) for i in range(num_labels)}

    _, attn = get_cls_attention(text)
    n       = len([t for t in tokens if t != "[PAD]"])
    tokens  = tokens[:n]
    attn    = attn[:n]

    is_content = np.array([
        t not in ("[CLS]", "[SEP]", "[PAD]") and not t.startswith("##")
        for t in tokens
    ])

    ranked     = np.argsort(attn * is_content)[::-1]
    key_tokens = [
        tokens[i]
        for i in ranked[:top_k]
        if attn[i] >= attn_threshold and is_content[i]
    ]

    escalate = (pred_id == 0) and (conf >= 0.80)

    return {
        "label":      label,
        "confidence": round(conf, 4),
        "all_probs":  all_probs,
        "key_tokens": key_tokens,
        "escalate":   escalate,
    }


print("analyze_tweet() is ready.")

In [ ]:
test_cases = [
    (
        "I have been waiting 3 weeks for a refund and nobody is helping me. This is absolutely unacceptable!",
        "Expected: Negative + escalate=True"
    ),
    (
        "My order arrived yesterday. No issues so far, tracking worked fine.",
        "Expected: Neutral"
    ),
    (
        "The support agent was incredibly helpful and resolved everything in minutes. Love this service!",
        "Expected: Positive"
    ),
    (
        "The onboarding emails were confusing, but the agent fixed everything politely.",
        "Expected: Positive or Neutral (mixed signals)"
    ),
]

print("ANALYZE_TWEET() DEMO")
print("=" * 60)

for text, note in test_cases:
    result = analyze_tweet(text)
    print(f"\nInput: {text[:80]}")
    print(f"  ({note})")
    print(f"  Label      : {result['label']}")
    print(f"  Confidence : {result['confidence']:.3f}")
    print(f"  All probs  : {result['all_probs']}")
    print(f"  Key tokens : {result['key_tokens']}")
    print(f"  Escalate?  : {'YES' if result['escalate'] else 'No'}")
    print("-" * 60)

In [ ]:
neg_text     = test_cases[0][0]
tokens, attn = get_cls_attention(neg_text)
n       = len([t for t in tokens if t != "[PAD]"])
tokens  = tokens[:n]
attn    = attn[:n]

cmap   = plt.cm.YlOrRd
colors = [cmap(a) for a in attn]

fig, ax = plt.subplots(figsize=(14, 2.5))
for i, (tok, col, a) in enumerate(zip(tokens, colors, attn)):
    ax.text(
        i, 0.5, tok,
        color="black" if a < 0.7 else "white",
        fontsize=9, ha="center", va="center",
        bbox=dict(boxstyle="round,pad=0.3", facecolor=col, linewidth=0),
    )

ax.set_xlim(-0.5, len(tokens) - 0.5)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Token Highlight — Negative escalation example (darker = higher attention)", fontsize=11)
plt.tight_layout()
plt.show()

---
## Reflection Questions

Answer each question in your own words.

---

### Q1 — What lever (data cleaning, hyperparameters, more epochs) most improved results?

> **Your answer:**
> The most impactful lever was learning rate and warmup. Starting at 5e-5 with a 10% warmup prevented the early spike in validation loss. A third epoch gave only a marginal F1 gain (~0.01), suggesting the model had largely converged. Data cleaning (removing duplicate or very short tweets) would be the next useful step.

---

### Q2 — Where would you add guardrails before deploying this signal live?

> **Your answer:**
> 1. Confidence threshold gate: never act on predictions below 0.65 — route to a human instead.
> 2. Out-of-distribution monitoring: track input length and vocabulary overlap with training data.
> 3. Feedback loop: have agents label model errors weekly and retrain monthly.
> 4. PII scrubbing: strip names and order numbers before the text reaches the model.

---

### Q3 — Which stakeholders benefit the most?

> **Your answer:**
>
> | Stakeholder | Benefit |
> |-------------|--------|
> | Support Lead | Negative + high-confidence tickets surface automatically |
> | Product Manager | Aggregate sentiment by feature area to prioritise backlog |
> | Compliance Officer | Saved key tokens explain every escalation decision |
> | Customer | Angry customers reach a senior agent faster |

---
## Deliverable Checklist

| # | Item | Status |
|---|------|--------|
| 1 | `tweet_eval` loaded, class distribution plotted | done |
| 2 | 2 example tweets saved per label | done |
| 3 | Tokenization pipeline with 128-token limit | done |
| 4 | DistilBERT fine-tuned (3 epochs, lr=5e-5) | done |
| 5 | Best checkpoint saved | done |
| 6 | Accuracy + macro F1 on val and test splits | done |
| 7 | Confidence histogram | done |
| 8 | CLS attention heatmap for all 3 sentiment classes | done |
| 9 | `analyze_tweet()` returning label, confidence, key_tokens, escalate | done |
| 10 | Reflection questions answered | done |

---
## Extensions

- Freeze early layers: add `for param in model.distilbert.transformer.layer[:2].parameters(): param.requires_grad = False` before training.
- Temperature scaling: fit a single temperature parameter on the validation set to improve calibration.
- Multilingual: swap `distilbert-base-uncased` for `distilbert-base-multilingual-cased` and test on non-English tweets.
- SHAP values: install `shap` and run `shap.Explainer` for a more rigorous token attribution.